# Practical 5 — Species Classification

**Context:** MegaDetector detects *animal* — but not which species.
Species classification is the second stage of the camera trap pipeline:

```
images -> MegaDetector (detect animal) -> crops -> species classifier (this notebook)
```

We use **SpeciesNet** (Google) — a global classifier trained on 65M+ camera trap images,
covering 2000+ species with optional country-level geofencing.

| Classifier | Architecture | Species | Geographic scope |
|-----------|-------------|---------|------------------|
| **SpeciesNet** (Google) | EfficientNetV2-M, 480x480 | 2000+ | Global (with geofencing) |

**Learning goals:**
- Understand the two-stage pipeline: detection then classification
- Run SpeciesNet on Serengeti camera trap crops
- Evaluate classifier quality with precision, recall, and confusion matrix

**Prerequisites:** Run `practical_3_megadetector_legacy.ipynb` or
`practical_3_megadetector_ultralytics.ipynb` first to generate the animal crops.

---


## Environment Setup

```bash


```

In [1]:
# Colab only — install dependencies if not already available
# import sys

# !git clone  https://github.com/cwinkelmann/usde-innovations-applications-forest-it.git fit-course
# !cd fit-course && git pull
# !pip install -e "./fit-course[training,dev]"
#
# sys.path.append('./fit-course')

In [2]:
%matplotlib inline

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("../data")
IMG_DIR = DATA_DIR / "camera_trap" / "serengeti_subset"

# MegaDetector output from Practical 3 (try v1000 first, fall back to legacy)
MD_OUTPUT_V1000 = DATA_DIR / "megadetector_output_v1000"
MD_OUTPUT_LEGACY = DATA_DIR / "megadetector_output"

if (MD_OUTPUT_V1000 / "v1000_detections.json").exists():
    MD_OUTPUT = MD_OUTPUT_V1000
    MD_JSON = MD_OUTPUT / "v1000_detections.json"
    print(f"Using MD v1000 output: {MD_JSON}")
elif (MD_OUTPUT_LEGACY / "mdv5a_detections.json").exists():
    MD_OUTPUT = MD_OUTPUT_LEGACY
    MD_JSON = MD_OUTPUT / "mdv5a_detections.json"
    print(f"Using MDv5A legacy output: {MD_JSON}")
else:
    raise FileNotFoundError(
        "No MegaDetector output found. Run Practical 3 first."
    )

CROPS_DIR = MD_OUTPUT / "animal_crops"
OUTPUT_DIR = DATA_DIR / "species_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load MegaDetector JSON
with open(MD_JSON) as f:
    md_data = json.load(f)

crop_files = sorted(CROPS_DIR.glob("*.*"))
print(f"Animal crops: {len(crop_files)} in {CROPS_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")

FileNotFoundError: No MegaDetector output found. Run Practical 3 first.

In [ ]:
%matplotlib inline

# Show a few crops as reminder
n_show = min(8, len(crop_files))
fig, axes = plt.subplots(1, n_show, figsize=(16, 3))
if n_show == 1:
    axes = [axes]
for ax, cp in zip(axes, crop_files[:n_show]):
    ax.imshow(Image.open(cp))
    ax.set_title(cp.name, fontsize=6)
    ax.axis("off")
plt.suptitle("MegaDetector animal crops (input for species classifier)", fontsize=10)
plt.tight_layout()

---
## Part 1 — SpeciesNet (Global, 2000+ species)

SpeciesNet is Google's species classifier, trained on 65M+ camera trap images.
It uses EfficientNetV2-M (480x480 px input) and supports **geofencing** —
restricting predictions to species that actually occur in a given country.

Two usage modes:
- **Full pipeline** (`components="all"`): runs MegaDetector internally + classifies
- **Classifier only** (`components="classifier"`): uses pre-computed MegaDetector detections

We use **classifier-only mode** since we already have MegaDetector output from Practical 3.

### 1.1 — Load SpeciesNet

First run downloads the model weights from Kaggle (~500 MB). Subsequent runs use the cache.

In [ ]:
from speciesnet import SpeciesNet

# Load classifier-only mode — we already have MegaDetector detections
snet = SpeciesNet(
    "kaggle:google/speciesnet/pyTorch/v4.0.2a/1",
    components="classifier",
)
print("SpeciesNet classifier loaded")

### 1.2 — Classify with pre-computed detections

We pass the original images + the MegaDetector JSON detections.
SpeciesNet will crop each image using the bounding boxes and classify.

We set `country="TZA"` (Tanzania) for geofencing — this filters out
species that don't occur in the Serengeti.

In [ ]:
# Build the detections dict in SpeciesNet format
# SpeciesNet expects: {filepath: {"detections": [{"category": "1", "conf": 0.9, "bbox": [x,y,w,h]}]}}
detections_dict = {}
image_filepaths = []

for img_data in md_data["images"]:
    filepath = str(IMG_DIR / img_data["file"])
    if not Path(filepath).exists():
        continue
    # Only include images with animal detections
    animal_dets = [d for d in img_data["detections"] if d.get("category") == "1"]
    if animal_dets:
        detections_dict[filepath] = {"detections": animal_dets}
        image_filepaths.append(filepath)

print(f"Images with animal detections: {len(image_filepaths)}")

# Run classifier on original images using MD bounding boxes
snet_results = snet.classify(
    filepaths=image_filepaths,
    detections_dict=detections_dict,
    country="TZA", # Code in this format: https://en.wikipedia.org/wiki/ISO_3166-1_alpha-3
    batch_size=8,
    progress_bars=True,
)

n_preds = len(snet_results.get("predictions", []))
print(f"\nClassified {n_preds} images")

### 1.3 — Parse SpeciesNet results

SpeciesNet returns a taxonomy string like `mammalia;carnivora;felidae;panthera;panthera_leo;lion`.
We extract the common name (last element) for display.

In [ ]:
%matplotlib inline

snet_records = []
# predictions is a list of dicts, each with "filepath" and "classifications"
predictions = snet_results.get("predictions", [])

for pred in predictions:
    filepath = pred.get("filepath", "")
    filename = Path(filepath).name
    classifications = pred.get("classifications", {})
    classes = classifications.get("classes", [])
    scores = classifications.get("scores", [])

    if classes and scores:
        top_class = classes[0]
        top_score = scores[0]
        # Extract common name from taxonomy string
        # e.g. "mammalia;carnivora;felidae;panthera;panthera_leo;lion"
        parts = top_class.split(";")
        common_name = parts[-1].replace("_", " ") if parts else top_class
        species = parts[4].replace("_", " ") if len(parts) > 4 else common_name
    else:
        common_name = "unknown"
        species = "unknown"
        top_score = 0.0
        top_class = ""

    snet_records.append({
        "filename": filename,
        "speciesnet_species": species,
        "speciesnet_common": common_name,
        "speciesnet_conf": round(top_score, 4),
        "speciesnet_full_label": top_class,
    })

snet_df = pd.DataFrame(snet_records)
print(f"\nSpeciesNet predictions: {len(snet_df)}")
print(f"\nSpecies distribution:")
print(snet_df["speciesnet_common"].value_counts().to_string())

### 1.4 — Visualize SpeciesNet predictions

In [ ]:
# Show predictions on a grid of original images
n_show = min(9, len(snet_records))
cols = 3
rows = (n_show + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flat if n_show > 1 else [axes]

for i, ax in enumerate(axes):
    if i >= n_show:
        ax.axis("off")
        continue

    rec = snet_records[i]
    img_path = IMG_DIR / rec["filename"]
    if img_path.exists():
        ax.imshow(np.array(Image.open(img_path)))
    ax.set_title(
        f"{rec['speciesnet_common']} ({rec['speciesnet_conf']:.2f})\n{rec['filename']}",
        fontsize=8,
    )
    ax.axis("off")

plt.suptitle("SpeciesNet predictions (Serengeti, geofenced to Tanzania)", fontsize=12)
plt.tight_layout()


### 1.5 — Classify pre-cropped images directly

Instead of passing the original images with bounding boxes, we can also run
SpeciesNet directly on the pre-cropped animal patches.
This is useful when you only have crops and no original images or detection JSON.

In [ ]:
# Classify the saved crop files directly (no detections_dict)
crop_filepaths = [str(f) for f in sorted(CROPS_DIR.glob("*.*"))]
print(f"Classifying {len(crop_filepaths)} crop files from {CROPS_DIR}")

crop_results = snet.classify(
    filepaths=crop_filepaths,
    country="TZA",
    batch_size=8,
    progress_bars=True,
)

# Parse results
crop_records = []
for pred in crop_results.get("predictions", []):
    filepath = pred.get("filepath", "")
    filename = Path(filepath).name
    classifications = pred.get("classifications", {})
    classes = classifications.get("classes", [])
    scores = classifications.get("scores", [])
    if classes and scores:
        top_class = classes[0]
        top_score = scores[0]
        parts = top_class.split(";")
        common_name = parts[-1].replace("_", " ") if parts else top_class
    else:
        common_name = "unknown"
        top_score = 0.0
    crop_records.append({"filename": filename, "speciesnet_common": common_name, "speciesnet_conf": round(top_score, 4)})

crop_df = pd.DataFrame(crop_records)
print(f"\nCrop classification results: {len(crop_df)}")
print(crop_df["speciesnet_common"].value_counts().to_string())


In [ ]:
# Geofencing demo on HNEE German camera trap data (from Practical 3).
# Raw SpeciesNet scores already computed — we only apply the geofence filter.
# No re-detection, no re-classification.

from pathlib import Path
from speciesnet import SpeciesNet
from speciesnet.geofence_utils import geofence_animal_classification

COUNTRY = "DEU"
HNEE_RESULTS = DATA_DIR / "megadetector_output_hnee" / "speciesnet_hnee_p02_test.json"

snet_ens = SpeciesNet(
    "kaggle:google/speciesnet/pyTorch/v4.0.2a/1",
    components="ensemble",
)
taxonomy_map = snet_ens.ensemble.taxonomy_map
geofence_map = snet_ens.ensemble.geofence_map

with open(HNEE_RESULTS) as f:
    hnee_data = json.load(f)

rows = []
for pred in hnee_data.get("predictions", []):
    filename = Path(pred["filepath"]).name
    classes = pred.get("classifications", {}).get("classes", [])
    scores  = pred.get("classifications", {}).get("scores", [])
    if not classes:
        continue

    raw_label = classes[0].split(";")[-1].replace("_", " ")

    label_geo, score_geo, _ = geofence_animal_classification(
        labels=classes, scores=scores,
        country=COUNTRY, admin1_region=None,
        taxonomy_map=taxonomy_map, geofence_map=geofence_map,
        enable_geofence=True,
    )
    geo_label = label_geo.split(";")[-1].replace("_", " ")

    rows.append({
        "file": filename,
        "without geofencing": raw_label,
        f"with {COUNTRY} geofencing": geo_label,
        "changed": raw_label != geo_label,
    })

df = pd.DataFrame(rows)
print(f"HNEE German camera traps — {COUNTRY} geofencing effect\n")
print(df[["file", "without geofencing", f"with {COUNTRY} geofencing"]].to_string(index=False))
print(f"\n{df['changed'].sum()}/{len(df)} predictions changed")
print("(e.g. white-tailed deer / elk → red deer: non-native North American species filtered out)")


In [ ]:
snet_ens.ensemble.geofence_map


In [ ]:
# Apply geofencing post-hoc — no re-detection or re-classification.
# Loads only the ensemble component (geofence + taxonomy maps) and
# post-processes the raw classifier scores from cell 15.
#
# What geofencing does:
#   The model sees visual features and ranks all known species by confidence.
#   Geofencing checks: "does the top-ranked species actually occur in this country?"
#   If not, it rolls the prediction UP the taxonomy tree until it finds a level
#   that IS valid in that country — or stops at class/kingdom.
#
#   Result: instead of confidently predicting an IMPOSSIBLE species, the model
#   gives an HONEST answer at a coarser level.
#   "thomson's gazelle in Germany" → "Bovidae family" (something bovid-looking,
#    but no bovid native to Germany matches — so we can't be more specific).

from speciesnet import SpeciesNet
from speciesnet.geofence_utils import geofence_animal_classification

COUNTRY = "DEU"  # ← ISO 3166-1 alpha-3 code

snet_ens = SpeciesNet(
    "kaggle:google/speciesnet/pyTorch/v4.0.2a/1",
    components="ensemble",
)
taxonomy_map = snet_ens.ensemble.taxonomy_map
geofence_map = snet_ens.ensemble.geofence_map
print(f"Geofence maps loaded — applying country={COUNTRY}\n")

rows = []
for pred in crop_results.get("predictions", []):
    filename = Path(pred.get("filepath", "")).name
    classes = pred.get("classifications", {}).get("classes", [])
    scores  = pred.get("classifications", {}).get("scores", [])
    if not classes:
        continue

    raw_label = classes[0].split(";")[-1].replace("_", " ")

    label_geo, score_geo, source = geofence_animal_classification(
        labels=classes, scores=scores,
        country=COUNTRY, admin1_region=None,
        taxonomy_map=taxonomy_map, geofence_map=geofence_map,
        enable_geofence=True,
    )
    geo_label = label_geo.split(";")[-1].replace("_", " ")

    rows.append({
        "filename": filename,
        "without_geofencing": raw_label,
        f"with_{COUNTRY}_geofencing": geo_label,
        "rolled_up": raw_label != geo_label,
    })

df_cmp = pd.DataFrame(rows)
n_changed = df_cmp["rolled_up"].sum()
print(f"{n_changed}/{len(df_cmp)} predictions rolled up (species absent from {COUNTRY})\n")

print("Without geofencing → with geofencing (unique pairs):")
print("(rolled-up predictions are HONEST — the model refuses to name an impossible species)\n")
changed = (
    df_cmp[df_cmp["rolled_up"]][["without_geofencing", f"with_{COUNTRY}_geofencing"]]
    .drop_duplicates()
    .rename(columns={"without_geofencing": "raw prediction", f"with_{COUNTRY}_geofencing": f"after {COUNTRY} geofencing"})
)
print(changed.to_string(index=False))


In [ ]:
# Show 2 example crops per predicted class
from collections import defaultdict

# Group records by predicted class, keep up to 2 per class
by_class = defaultdict(list)
for rec in crop_records:
    cls = rec["speciesnet_common"]
    if len(by_class[cls]) < 2:
        by_class[cls].append(rec)

# Flatten: sorted by class name, 2 columns per class
sample_records = [rec for cls in sorted(by_class) for rec in by_class[cls]]

cols = 4  # 2 examples × 2 classes side by side
rows = (len(sample_records) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
axes = axes.flat if len(sample_records) > 1 else [axes]

for i, ax in enumerate(axes):
    if i >= len(sample_records):
        ax.axis("off")
        continue
    rec = sample_records[i]
    img_path = CROPS_DIR / rec["filename"]
    if img_path.exists():
        ax.imshow(np.array(Image.open(img_path)))
    conf = rec["speciesnet_conf"]
    color = "green" if conf >= 0.7 else ("orange" if conf >= 0.4 else "red")
    ax.set_title(f"{rec['speciesnet_common']}\n{conf:.2f}", fontsize=8, color=color)
    ax.axis("off")

plt.suptitle(f"SpeciesNet on crops — 2 examples per class ({len(by_class)} classes)", fontsize=11)
plt.tight_layout()
plt.show()


---
## Part 2 — Save & Review Results

Save SpeciesNet predictions to CSV and review the confidence distribution.

In [ ]:
merged = snet_df.copy()

# Save results
merged.to_csv(OUTPUT_DIR / "species_predictions.csv", index=False)

print(f"Predictions: {len(merged)} images")
print(f"Saved -> {OUTPUT_DIR / 'species_predictions.csv'}")
print()

display_cols = ["filename", "speciesnet_common", "speciesnet_conf"]
merged[display_cols].head(15)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(merged["speciesnet_conf"].dropna(), bins=20, color="steelblue", edgecolor="white")
ax.set_xlabel("Confidence")
ax.set_ylabel("Count")
ax.set_title("SpeciesNet confidence distribution (Serengeti, geofenced to Tanzania)")
plt.tight_layout()
plt.show()


---

## Part 3 — Evaluating Classifier Results

A model that reports "92% accuracy" sounds good — but on a dataset
where 92% of images are one class, a model that always predicts that
class also gets 92%. Accuracy alone is not enough.

| Metric | Formula | When it matters |
|--------|---------|----------------|
| **Precision** | TP / (TP + FP) | Cost of false alarms is high |
| **Recall** | TP / (TP + FN) | Cost of missing cases is high |
| **F1** | 2 x P x R / (P + R) | Balance of both |

We compare SpeciesNet predictions against the ground truth species labels
from the Serengeti metadata (downloaded in Practical 1).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Load ground truth from Serengeti metadata
META_PATH = DATA_DIR / "camera_trap" / "serengeti_meta.json"

if META_PATH.exists() and len(snet_df) > 0:
    with open(META_PATH) as f:
        meta = json.load(f)
    
    cat_map = {c["id"]: c["name"] for c in meta["categories"]}
    img_id_to_file = {img["id"]: Path(img["file_name"]).name for img in meta["images"]}
    img_id_to_species = {}
    for a in meta["annotations"]:
        species = cat_map.get(a["category_id"], "unknown")
        if species != "empty":
            img_id_to_species[a["image_id"]] = species
    
    file_to_gt = {img_id_to_file[iid]: sp for iid, sp in img_id_to_species.items()
                  if iid in img_id_to_file}
    
    eval_rows = []
    for _, row in snet_df.iterrows():
        gt = file_to_gt.get(row["filename"])
        if gt and row["speciesnet_common"] != "blank":
            eval_rows.append({
                "true_label": gt,
                "pred_label": row["speciesnet_common"],
                "confidence": row["speciesnet_conf"],
            })
    
    eval_df = pd.DataFrame(eval_rows)
    print(f"Evaluation set: {len(eval_df)} images with GT + predictions")
else:
    print("No evaluation data available. Make sure you ran Practical 1 and have predictions.")

y_true = eval_df["true_label"].tolist()
y_pred = eval_df["pred_label"].tolist()

### 3.1 — Classification report

`sklearn.metrics.classification_report` gives per-class precision, recall, and F1.

In [ ]:
print(classification_report(y_true, y_pred, digits=3, zero_division=0))

### 3.2 — Confusion matrix

Rows = true class, columns = predicted class. Off-diagonal entries are errors.
Large off-diagonal values tell you which pairs of classes the model confuses most.

In [ ]:
labels = sorted(set(y_true + y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.8), max(5, len(labels) * 0.7)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=45)
ax.set_title("Confusion matrix — SpeciesNet on Serengeti crops")
plt.tight_layout()

### 3.3 — Class imbalance awareness

If the dataset is imbalanced, a model that always predicts the majority class
will look good on accuracy but terrible on per-class recall.

In [ ]:
class_counts = eval_df["true_label"].value_counts()
majority_class = class_counts.idxmax()
majority_accuracy = class_counts.max() / len(eval_df)

print("Class distribution in evaluation set:")
print(class_counts.to_string())
print(f"\nA model that always predicts '{majority_class}' achieves:")
print(f"  Accuracy = {majority_accuracy:.1%}")
print(f"  Recall for all other classes = 0%")
print(f"\nThis is why F1 and per-class recall matter more than overall accuracy.")

---
## Exercises

1. **Geofencing** — Re-run SpeciesNet without geofencing (`country=None`).
   Do the predictions change? What impossible species appear?

2. **Confidence threshold** — What percentage of SpeciesNet predictions have
   confidence > 0.8? What does the confidence distribution tell you about how
   well the model matches the Serengeti domain?

3. **Crops vs full images** — Compare the results from section 1.2
   (original images + bounding boxes) with section 1.5 (crops directly).
   Do you get the same predictions? Why or why not?

4. **Failure inspection** — Find 3 images where SpeciesNet is wrong or uncertain
   (confidence < 0.5). What do these images have in common?


## Reflection

- **Global vs specialist models**: SpeciesNet covers 2000+ species globally.
  When might a regional specialist model perform better? What trade-offs
  would you accept — breadth vs depth?

- **Geofencing as guardrail**: SpeciesNet uses country-level geofencing to
  prevent impossible predictions. Is this always a good idea? What if an
  invasive species appears? What if the GPS metadata is wrong?

- **The two-stage pipeline**: MegaDetector (detect) then SpeciesNet (classify)
  is the standard camera trap workflow. What are the advantages of splitting
  detection and classification? Could you train a single end-to-end model instead?
  What would you gain and lose?

- **Crops vs context**: In section 1.5 you classified crops without the surrounding
  scene. Does removing habitat context change the predictions? What does this
  tell you about what the model has learned?
